In [ ]:
from pathlib import Path
import pandas as pd
import pyomo.environ as pyo


def build_max_covering_model(
    matrix_path,
    population_path=None,
    max_cover_dist_m=10_000,
    p=10,
    target_col="target_id",
    source_col="source_id",
    distance_col="total_dist",
    population_id_col="ID",
    population_weight_col="population",
):
    # Load sparse distance matrix
    matrix = pd.read_parquet(matrix_path)

    # Keep only source-target pairs within the coverage threshold
    covered_pairs = matrix.loc[
        matrix[distance_col] <= max_cover_dist_m,
        [target_col, source_col],
    ].drop_duplicates()

    targets = sorted(matrix[target_col].unique())
    sources = sorted(matrix[source_col].unique())

    # Target weights
    if population_path is None:
        weights = {i: 1.0 for i in targets}
    else:
        population = pd.read_parquet(population_path)
        weights = (
            population
            .set_index(population_id_col)[population_weight_col]
            .reindex(targets)
            .fillna(0.0)
            .astype(float)
            .to_dict()
        )

    # Coverage neighborhood: N[i] = sources that can cover target i
    cover_sources = (
        covered_pairs
        .groupby(target_col)[source_col]
        .apply(list)
        .to_dict()
    )

    # Pyomo model
    model = pyo.ConcreteModel()

    model.I = pyo.Set(initialize=targets)   # demand / population points
    model.J = pyo.Set(initialize=sources)   # candidate / source facilities

    model.w = pyo.Param(model.I, initialize=weights, default=0.0)

    model.x = pyo.Var(model.J, domain=pyo.Binary)  # 1 if source j is selected
    model.y = pyo.Var(model.I, domain=pyo.Binary)  # 1 if target i is covered

    # Select exactly p facilities
    model.facility_budget = pyo.Constraint(
        expr=sum(model.x[j] for j in model.J) == p
    )

    # A target can be covered only if at least one covering source is selected
    def coverage_rule(m, i):
        eligible = cover_sources.get(i, [])
        if not eligible:
            return m.y[i] == 0
        return m.y[i] <= sum(m.x[j] for j in eligible)

    model.coverage = pyo.Constraint(model.I, rule=coverage_rule)

    # Maximize covered population
    model.objective = pyo.Objective(
        expr=sum(model.w[i] * model.y[i] for i in model.I),
        sense=pyo.maximize,
    )

    return model


In [ ]:
data = Path(r'C:\local\Download_Depot\east-timor_data\outputs')

matrix_path = data / r"distance_matrix_pop_1_sample_1_max_none_agg_none_maxdist_none_with_secondary_tags_amenity_all_candidates_spacing_5000_maxsnap_5000.parquet"
population_path = data / r"population_pop_1_sample_1_max_none_agg_none_maxdist_none_with_secondary_tags_amenity_all_candidates_spacing_5000_maxsnap_5000.parquet"

model = build_max_covering_model(
    matrix_path=matrix_path,
    population_path=population_path,
    max_cover_dist_m=10_000,  # 10 km
    p=20,
)

solver = pyo.SolverFactory("gurobi")  # or "cbc", "gurobi", "glpk"
result = solver.solve(model, tee=True)

selected_sources = [
    j for j in model.J
    if pyo.value(model.x[j]) > 0.5
]

covered_targets = [
    i for i in model.I
    if pyo.value(model.y[i]) > 0.5
]

covered_population = pyo.value(model.objective)

print("Selected sources:", selected_sources)
print("Covered targets:", len(covered_targets))
print("Covered population:", covered_population)


In [ ]:
from distance_pipeline.viz import build_solution_layers, plot_max_cover_solution


In [ ]:
base_path_for_document = Path(r'C:\Users\joaqu\Dropbox\Apps\Overleaf\Real Life Distance Generator')
figures_path = base_path_for_document / 'figures'
optimization_figure_path = figures_path / 'timor_leste_max_cover_solution.png'

existing_sources_path = data / r'existing_sources_pop_1_sample_1_max_none_agg_none_maxdist_none_with_secondary_tags_amenity_all_candidates_spacing_5000_maxsnap_5000.parquet'

solution = build_solution_layers(
    model=model,
    matrix_path=matrix_path,
    population_path=population_path,
    sources_path=existing_sources_path,
    max_cover_dist_m=10_000,
)

fig, ax = plot_max_cover_solution(
    solution,
    title='Timor-Leste maximum covering solution, 10 km threshold',
    output_path=optimization_figure_path,
)
